# Cost Analysis

Estimates cost per valid requirement-matching failure. This is an analysis over paper result metrics, not a GPU benchmark.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

for module in [
    "shared",
    "mnist",
    "precondition_validity",
    "cost_analysis",
    "mnist_lora_ablation",
    "shared_lora_pilot",
]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "cost-analysis"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from cost_analysis import build_rows, write_results
from shared import requirement_rows

results_path, summary_path = write_results(ROOT)
valid_rows, cost_rows = build_rows(requirement_rows(ROOT))

print("largest estimated valid-failure yields")
for row in valid_rows[:8]:
    print(
        f"{row['dataset']} {row['requirement']} {row['method']} | {float(row['estimated_valid_failures']):.1f}"
    )

print()
print("cheapest estimated valid failures")
print("dataset req method | estimated valid failures | cost | cost/estimated failure")
for row in cost_rows[:12]:
    print(
        f"{row['dataset']} {row['requirement']} {row['method']} | "
        f"{float(row['estimated_valid_failures']):.1f} | ${float(row['estimated_cost_usd']):.2f} | "
        f"${float(row['cost_per_estimated_valid_failure_usd']):.2f}"
    )
print(results_path)
print(summary_path)

Rows without both pass-rate and precondition-match data are excluded.